# Yield, toy capability, and pass/fail association

This notebook follows Days 15–17. The label-based yield summary and sensor associations use observed SECOM data. Cp/Cpk uses a separate hypothetical measurement with explicitly invented teaching specifications. These analyses answer different questions.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'data').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
from src.secom_data import load_secom
from src.quality import summarize_yield, capability_summary
from src.association import compare_pass_fail

## 1. Estimate pass fraction and uncertainty

The source encodes pass as -1 and fail as +1. The Wilson interval is a binomial sampling interval; it does not correct for unknown sampling bias or dependence.

In [ ]:
data = load_secom(ROOT / 'data' / 'raw')
yield_result = summarize_yield(data.labels)
print(yield_result)
print(f'Pass fraction: {yield_result.yield_fraction:.3%}')
print(f'{yield_result.confidence:.0%} Wilson interval: {yield_result.ci_low:.3%}–{yield_result.ci_high:.3%}')

## 2. Cp/Cpk in a separate toy example

These are made-up measurements and specifications, solely to learn the formulas. They are not SECOM features or fab specifications. Cp/Cpk describe capability only when the measured process is stable and specifications are genuine.

In [ ]:
toy_measurements = [9.82, 10.04, 10.11, 9.96, 10.07, 9.91, 10.02, 10.13, 9.88, 10.06, 9.97, 10.09]
toy_lsl, toy_usl = 9.5, 10.5  # invented teaching limits, in arbitrary units
toy_capability = capability_summary(toy_measurements, toy_lsl, toy_usl)
print('SIMULATED TEACHING EXAMPLE ONLY')
print(toy_capability)

## 3. Compare sensor distributions by label

Welch tests are univariate. Hedges g gives a standardized difference (fail mean minus pass mean). Benjamini–Hochberg q-values adjust the family of tested features. This is association analysis only; labels do not select variables for the PCA monitoring notebook.

In [ ]:
association = compare_pass_fail(data.features, data.labels)
eligible = association['p_value_welch'].notna().sum()
significant = association['q_value_bh'] < 0.05
print(f'Features tested with sufficient observations: {eligible} / {data.features.shape[1]}')
print(f'Features with BH q < 0.05: {int(significant.sum())}')
display(association.head(10))

## Explain before moving on

1. Why is the pass fraction not a capability index?
2. Why are the capability values clearly labeled simulated?
3. What does Hedges g add beyond a p-value?
4. What does a BH-adjusted q-value account for, and what causal claim can it not support?